# AI-Augmented Threat Hunting System for Real-Time Detection of Polymorphic Malware in Enterprise Networks

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
!pip install scikit-learn matplotlib seaborn torch pandas
#Dependencies


In [ ]:
!pip install scikit-learn matplotlib seaborn torch pandas keras

import random
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

api_calls = [
    "CreateFile", "ReadFile", "WriteFile", "DeleteFile", "OpenProcess",
    "CreateRemoteThread", "VirtualAllocEx", "RegOpenKey", "RegSetValue",
    "ConnectNamedPipe", "GetProcAddress", "LoadLibrary", "HttpSendRequest",
    "NtQuerySystemInformation", "SetWindowsHookEx"
]

def generate_sample(is_malicious):
    length = random.randint(10, 30)
    behavior = []
    for _ in range(length):
        if is_malicious:
            call = random.choices(api_calls, weights=[1,1,1,2,3,4,4,3,3,3,2,2,3,3,4])[0]
        else:
            call = random.choices(api_calls, weights=[3,3,3,1,1,1,1,2,2,1,1,1,1,1,1])[0]
        behavior.append(call)
    return behavior

data, labels = [], []
for _ in range(1000):
    label = random.choice([0, 1])
    sample = generate_sample(label)
    data.append(sample)
    labels.append(label)

df = pd.DataFrame({'api_calls': data, 'label': labels})

all_calls = [call for sublist in df['api_calls'] for call in sublist]
le = LabelEncoder()
le.fit(all_calls)

encoded_sequences = [le.transform(seq) for seq in df['api_calls']]
max_len = max(len(seq) for seq in encoded_sequences)
padded_sequences = pad_sequences(encoded_sequences, maxlen=max_len)

X = np.array(padded_sequences)
y = np.array(df['label'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = torch.tensor(X_train, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        out = self.fc(hidden[-1])
        return self.softmax(out)

vocab_size = len(le.classes_)
model = LSTMClassifier(vocab_size, embedding_dim=64, hidden_dim=128, output_dim=2)

train_data = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

loss_fn = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(5):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(X_batch)
        loss = loss_fn(output, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {epoch_loss:.4f}")

model.eval()
with torch.no_grad():
    y_pred = model(X_test.to(device)).argmax(dim=1).cpu().numpy()

acc = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {acc:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Benign', 'Malicious'], yticklabels=['Benign', 'Malicious'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()
